# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** ⚙️ **Bản LOCAL** — Jupyter Lab/Notebook local (Python 3.10+) + Neo4j AuraDB/local, dùng file `.env` thay cho Colab Secrets. (Bản gốc chạy trên Colab: `Day19_GraphRAG_vs_FlatRAG_Production_Lab_Guide.ipynb`)  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

### Secrets — chạy LOCAL bằng file `.env`
Đây là bản đã chỉnh để chạy trên máy local (Jupyter Lab/Notebook), không dùng Colab Secrets.

Các bước:
1. `pip install -r requirements.txt` (đã gồm `python-dotenv`).
2. `cp .env.example .env` rồi điền các giá trị thật vào `.env`.
3. Notebook sẽ tự nạp `.env` qua `python-dotenv` ở cell 1.2 — không cần Colab Secrets.

Biến cần khai báo trong `.env`:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

**Không hard-code API key vào notebook nộp bài** — `.env` đã nằm trong `.gitignore`, không bị commit lên GitHub.

In [1]:
#@title 1.1 — Install (auto-skip nếu chạy local đã `pip install -r requirements.txt`)
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx datasets python-dotenv
else:
    print(
        "Local run: dependencies nên được cài trước bằng:\n"
        "    pip install -r requirements.txt\n"
        "Bỏ qua %pip install trong notebook để tránh cài lại/conflict môi trường local."
    )


Local run: dependencies nên được cài trước bằng:
    pip install -r requirements.txt
Bỏ qua %pip install trong notebook để tránh cài lại/conflict môi trường local.


In [22]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

# --- LOCAL RUN: nạp biến môi trường từ .env (bỏ qua nếu không có file .env / không cài dotenv) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("[Cảnh báo] python-dotenv chưa cài — chạy `pip install python-dotenv` hoặc set biến môi trường thủ công.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    # Local run: ưu tiên biến môi trường (nạp từ .env ở trên).
    # Nếu chạy lại trên Colab, fallback sang Colab Secrets khi biến chưa có trong os.environ.
    value = os.environ.get(name)
    if value is not None:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return default

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")

# NER+RE extraction (cell 2.1): openai | groq
EXTRACT_PROVIDER = get_secret("EXTRACT_PROVIDER", "openai").lower()
EXTRACT_MODEL = get_secret("EXTRACT_MODEL", "gpt-4o-mini")

# Answer generation + seed extraction (cell 3.2, 3.4, 4.3): openai | groq
GENERATE_PROVIDER = get_secret("GENERATE_PROVIDER", EXTRACT_PROVIDER).lower()
GENERATE_MODEL = get_secret("GENERATE_MODEL", EXTRACT_MODEL)

HF_TOKEN = get_secret("HF_TOKEN", "")

# LOCAL RUN: dữ liệu lưu trong thư mục dự án thay vì /content (Colab)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
DATA_PATH = "data/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong file `.env` (local) hoặc **Colab Secrets** (nếu chạy trên Colab). Không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `data/hackernoon_subset.csv` (thư mục `data/` trong repo), nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
# LOCAL RUN: ghi vào thư mục data/ trong repo thay vì /content (Colab)
os.makedirs("data", exist_ok=True)
OUTPUT_CSV = "data/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
# LIMIT_ROWS = 1_000_000
LIMIT_ROWS = 10_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ .env (local) hoặc Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào file .env (biến HF_TOKEN=hf_...) "
        "hoặc Colab Secrets nếu chạy trên Colab."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/HackerNoon/tech-company-news-data-dump/resolve/cc6144ccce683dcebb6e63f9a50ca084544af1c0/cleanedCompanyNews.csv
Retrying in 1s [Retry 1/5].


Đang ghi dữ liệu vào: data/hackernoon_subset.csv


Đang tải (MB):   2%|▏         | 5.79/300 [00:00<00:14, 20.51MB/s]         


[DỪNG] Đã đạt giới hạn số dòng: 10,000 dòng (Dung lượng: 5.79 MB)
✅ Hoàn thành: d:\AIThucChien\thuchanh\Day19-CaNhan3\Day19-2A202601376-TrinhHoangNam\data\hackernoon_subset.csv
   Rows: 10,000
   Size: 5.79 MB


In [28]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.
✅ Schema ready.


In [3]:
#@title 1.5 — Loader + exact dedup + chunking
# Bonus Near-Dedup: SimHash + LSH banding (Challenge A)
NEAR_DEDUP_ENABLED = True
NEAR_DEDUP_HAMMING_THRESHOLD = 3   # max bit differences (of 64) to treat as near-duplicate
NEAR_DEDUP_NUM_BANDS = 4             # LSH bands -> candidate pairs only within buckets

def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def _token_features(text):
    words = norm_space(text).lower().split()
    features = set(words)
    for i in range(len(words) - 1):
        features.add(f"{words[i]} {words[i+1]}")
    return features

def simhash64(text):
    v = [0] * 64
    for token in _token_features(text):
        h = int(hashlib.md5(token.encode("utf-8", errors="ignore")).hexdigest(), 16)
        for i in range(64):
            v[i] += 1 if (h >> i) & 1 else -1
    fp = 0
    for i, bit in enumerate(v):
        if bit >= 0:
            fp |= 1 << i
    return fp

def hamming64(a, b):
    return (a ^ b).bit_count()

def _simhash_band_keys(fingerprint, num_bands=4):
    band_size = 64 // num_bands
    mask = (1 << band_size) - 1
    return [(b, (fingerprint >> (b * band_size)) & mask) for b in range(num_bands)]

def near_dedup_simhash_lsh(df, threshold=NEAR_DEDUP_HAMMING_THRESHOLD, num_bands=NEAR_DEDUP_NUM_BANDS):
    """Near-dedup via SimHash + LSH. O(n) hashing + near-linear candidate checks (not O(n²) all-pairs)."""
    if len(df) <= 1:
        return df.copy(), pd.DataFrame(columns=[
            "kept_article_id", "removed_article_id", "hamming_distance",
            "kept_title", "removed_title", "merge_reason",
        ])

    work = df.reset_index(drop=True).copy()
    fingerprints = [
        simhash64(f"{t}\n{x}") for t, x in zip(work["title"], work["text"])
    ]

    # LSH: only compare articles that collide in at least one band.
    buckets = {}
    for idx, fp in enumerate(fingerprints):
        for key in _simhash_band_keys(fp, num_bands):
            buckets.setdefault(key, []).append(idx)

    uf_parent = list(range(len(work)))

    def find(x):
        while uf_parent[x] != x:
            uf_parent[x] = uf_parent[uf_parent[x]]
            x = uf_parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            uf_parent[rb] = ra

    audit_rows = []
    seen_pairs = set()
    for members in buckets.values():
        if len(members) < 2:
            continue
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                a, b = members[i], members[j]
                pair = (min(a, b), max(a, b))
                if pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                dist = hamming64(fingerprints[a], fingerprints[b])
                if dist <= threshold:
                    union(a, b)
                    kept_idx, removed_idx = (a, b) if len(work.at[a, "text"]) >= len(work.at[b, "text"]) else (b, a)
                    audit_rows.append({
                        "kept_article_id": work.at[kept_idx, "article_id"],
                        "removed_article_id": work.at[removed_idx, "article_id"],
                        "hamming_distance": dist,
                        "kept_title": work.at[kept_idx, "title"][:120],
                        "removed_title": work.at[removed_idx, "title"][:120],
                        "merge_reason": f"simhash_lsh: hamming<={threshold}",
                    })

    clusters = {}
    for idx in range(len(work)):
        root = find(idx)
        clusters.setdefault(root, []).append(idx)

    keep_indices = []
    for members in clusters.values():
        best = max(members, key=lambda i: (len(work.at[i, "text"]), -i))
        keep_indices.append(best)

    kept = work.iloc[sorted(keep_indices)].reset_index(drop=True)
    audit_df = pd.DataFrame(audit_rows)
    return kept, audit_df

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    # HackerNoon HF dump uses `description` as the main text field (not `body`/`story`).
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid", "url"], required=False)

    df = pd.DataFrame()
    text = raw[text_col].fillna("").map(norm_space)
    title = raw[title_col].fillna("").map(norm_space) if title_col else pd.Series([""] * len(raw), index=raw.index)
    # When only meta description is available, prepend title for richer chunk context.
    if text_col.lower() == "description" and title_col:
        df["text"] = (title + ". " + text).map(norm_space).str.strip(". ")
    else:
        df["text"] = text
    df["title"] = title

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if NEAR_DEDUP_ENABLED and len(df) > 1:
        before_near = len(df)
        df, near_dedup_audit_df = near_dedup_simhash_lsh(df)
        removed = before_near - len(df)
        print(
            f"Near dedup (SimHash+LSH, hamming<={NEAR_DEDUP_HAMMING_THRESHOLD}): "
            f"{before_near:,} -> {len(df):,} (-{removed:,})"
        )
        if len(near_dedup_audit_df):
            os.makedirs("outputs", exist_ok=True)
            near_dedup_audit_df.to_csv("outputs/near_dedup_audit.csv", index=False)
            print(f"Near-dedup audit: {len(near_dedup_audit_df):,} merged pairs -> outputs/near_dedup_audit.csv")
        else:
            near_dedup_audit_df = pd.DataFrame()
            print("Near-dedup audit: 0 merged pairs (threshold may be strict for this subset).")
    else:
        near_dedup_audit_df = pd.DataFrame()

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df, near_dedup_audit_df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df, near_dedup_audit_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())
if NEAR_DEDUP_ENABLED and len(near_dedup_audit_df):
    display(near_dedup_audit_df.head(10))

Exact dedup: 4,768 -> 4,104
Near dedup (SimHash+LSH, hamming<=3): 4,104 -> 4,098 (-6)
Near-dedup audit: 6 merged pairs -> outputs/near_dedup_audit.csv


Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 45449.63it/s]


,chunk_id,article_id,title,published_date,text
0,https://www.businesswire.com/news/home/20230515005855/en/onsemi-and-Sineng-Electric-Spearhead-the-Development-of-Sus...,https://www.businesswire.com/news/home/20230515005855/en/onsemi-and-Sineng-Electric-Spearhead-the-Development-of-Sus...,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications. (Nasdaq: ON) a leader in in...
1,https://www.wku.edu/healthinformationmanagement/::c0000,https://www.wku.edu/healthinformationmanagement/,Bachelor of Science in Health Information Management,2023-08-16,Bachelor of Science in Health Information Management. Health information management (HIM) is a diverse yet evolving ...
2,https://www.federalregister.gov/agencies/office-of-government-information-services::c0000,https://www.federalregister.gov/agencies/office-of-government-information-services,Office of Government Information Services,2023-09-27,Office of Government Information Services. We are announcing 2019''s annual Chief FOIA Officers'' Council meeting co...
3,https://www.buffalo.edu/administrative-services/information-for-suppliers.html::c0000,https://www.buffalo.edu/administrative-services/information-for-suppliers.html,Information for Suppliers,2023-07-03,Information for Suppliers. or service that use covered telecommunications equipment or services as a substantial or ...
4,https://www.devdiscourse.com/article/technology/2518545-ceinsys-tech-ltd-a-specialized-gis-mobility-engineering-serv...,https://www.devdiscourse.com/article/technology/2518545-ceinsys-tech-ltd-a-specialized-gis-mobility-engineering-serv...,Ceinsys Tech Ltd: A specialized GIS & Mobility engineering services provider celebrates 25 Years of Enhancing Possib...,2023-07-11,Ceinsys Tech Ltd: A specialized GIS & Mobility engineering services provider celebrates 25 Years of Enhancing Possib...


,kept_article_id,removed_article_id,hamming_distance,kept_title,removed_title,merge_reason
0,https://www.joplinglobe.com/region/national_business/samsung-showcases-groundbreaking-logic-innovations-at-system-ls...,https://www.galvnews.com/news_ap/business/samsung-showcases-groundbreaking-logic-innovations-at-system-lsi-tech-day-...,1,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023,simhash_lsh: hamming<=3
1,https://www.businesswireindia.com/esri-signs-agreement-with-malta-providing-access-to-gis-technology-training-83101....,https://www.finanznachrichten.de/nachrichten-2023-02/58332212-esri-signs-agreement-with-malta-providing-access-to-gi...,2,Esri Signs Agreement with Malta Providing Access to GIS Technology Training,Esri Signs Agreement with Malta Providing Access to GIS Technology Training,simhash_lsh: hamming<=3
2,https://www.vanguardngr.com/2023/06/cupp-to-service-chiefs-adopt-technology-strategy-to-tackle-nigerias-security-cha...,https://www.vanguardngr.com/2023/06/cupp-to-new-service-chiefs-adopt-technology-strategy-to-tackle-security-challenges/,1,CUPP to Service Chiefs: Adopt technology strategy to tackle Nigeria’s security challenges,CUPP to New Service Chiefs: Adopt technology strategy to tackle security challenges,simhash_lsh: hamming<=3
3,https://abc7news.com/411-out-of-service-att-customers-landlines/12671256/,https://abc7chicago.com/411-out-of-service-att-customers-landlines/12671256/,3,411 phone number is going out of service for millions of Americans,411 is going out of service for millions of Americans,simhash_lsh: hamming<=3
4,https://www.usnews.com/news/world/articles/2023-05-10/czech-president-ukraine-could-have-our-l-159-jets,https://www.channelnewsasia.com/world/czech-president-ukraine-could-have-our-l-159-jets-3479446,3,Czech President: Ukraine Could Have Our L-159 Jets,Czech president: Ukraine could have our L-159 jets,simhash_lsh: hamming<=3
5,https://www.tmcnet.com/tmcnet/mobile-world-congress/news/2023/02/20/9762825.htm,https://it.tmcnet.com/news/2023/02/20/9762825.htm,3,1Fit Central Asia''s top all-sports unlimited fitness membership app comes to the UK,1Fit Central Asia''s top all-sports unlimited fitness membership app comes to the UK,simhash_lsh: hamming<=3


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [23]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
from openai import OpenAI

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def openai_chat(messages, model=None, json_mode=False, max_retries=6):
    if openai_client is None:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")
    model = model or EXTRACT_MODEL or JUDGE_MODEL or "gpt-4o-mini"
    if not model:
        raise RuntimeError("Thiếu model OpenAI (EXTRACT_MODEL hoặc JUDGE_MODEL).")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = openai_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            err = str(e).lower()
            # backoff dài hơn khi rate limit
            if attempt == max_retries - 1:
                break
            wait = min(60, (2 ** attempt) * 2 + random.random())
            if "rate limit" in err or "429" in err:
                wait = min(90, wait * 2)
            time.sleep(wait)
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

def openai_json(system, user, model=None):
    text, usage = openai_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

def llm_json(system, user, provider=None, model=None):
    """Router JSON LLM: provider='openai' | 'groq'."""
    provider = (provider or EXTRACT_PROVIDER or "openai").lower()
    if provider == "openai":
        return openai_json(system, user, model=model or EXTRACT_MODEL)
    if provider == "groq":
        return groq_json(system, user, model=model or GROQ_MODEL)
    raise ValueError("provider must be 'openai' or 'groq'")

def llm_chat(messages, provider=None, model=None, json_mode=False):
    """Router chat LLM cho answer generation (cell 3.4 / 4.3)."""
    provider = (provider or GENERATE_PROVIDER or EXTRACT_PROVIDER or "openai").lower()
    if provider == "openai":
        return openai_chat(
            messages,
            model=model or GENERATE_MODEL or EXTRACT_MODEL,
            json_mode=json_mode,
        )
    if provider == "groq":
        return groq_chat(messages, model=model or GROQ_MODEL, json_mode=json_mode)
    raise ValueError("provider must be 'openai' or 'groq'")

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [5]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = llm_json(COREF_SYSTEM, prompt, provider=EXTRACT_PROVIDER, model=EXTRACT_MODEL)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Coref: 100%|██████████| 80/80 [22:21<00:00, 16.76s/it]


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [9]:
#@title 2.1 — NER + RE extraction
EXTRACT_DEBUG = True          # bật log chi tiết khi debug
EXTRACT_LOG_FIRST_N_ERRORS = 5  # in tối đa N lỗi API đầu tiên
EXTRACT_BATCH_SLEEP = 0.5     # giây nghỉ giữa các batch (giảm rate limit OpenAI)

# Dùng OpenAI gpt-4o-mini (đọc từ cell 1.2: EXTRACT_PROVIDER / EXTRACT_MODEL)
# Đổi trong .env: EXTRACT_PROVIDER=openai, EXTRACT_MODEL=gpt-4o-mini

ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

IMPORTANT: copy chunk_id exactly from INPUT (do not shorten or rewrite URLs).

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return llm_json(EXTRACT_SYSTEM, prompt, provider=EXTRACT_PROVIDER, model=EXTRACT_MODEL)

def _parse_relations_from_item(item, meta, stats, triples):
    cid = item.get("chunk_id")
    if cid not in meta:
        stats["skipped_chunk_id"] += 1
        if EXTRACT_DEBUG and len(stats["chunk_id_mismatches"]) < 5:
            stats["chunk_id_mismatches"].append({
                "returned_chunk_id": cid,
                "expected_sample": next(iter(meta.keys()), None),
            })
        return

    for x in item.get("relations", []):
        stats["relations_raw"] += 1
        s, t = norm_space(x.get("source")), norm_space(x.get("target"))
        st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
        if not s or not t:
            stats["skipped_empty_entity"] += 1
            continue
        if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
            stats["skipped_node_type"] += 1
            if EXTRACT_DEBUG and len(stats["bad_schema_samples"]) < 5:
                stats["bad_schema_samples"].append({
                    "chunk_id": cid,
                    "source_type": st,
                    "target_type": tt,
                    "relation": rel,
                })
            continue
        if rel not in ALLOWED_RELATIONS:
            stats["skipped_relation"] += 1
            if EXTRACT_DEBUG and len(stats["bad_relation_samples"]) < 5:
                stats["bad_relation_samples"].append({
                    "chunk_id": cid,
                    "relation": rel,
                })
            continue
        triples.append({
            "source_raw": s,
            "source_type": st,
            "relation": rel,
            "target_raw": t,
            "target_type": tt,
            "source_chunk_id": cid,
            "published_date": meta[cid] or "",
            "evidence": norm_space(x.get("evidence")),
            "confidence": float(x.get("confidence") or 0.0),
        })
        stats["relations_kept"] += 1

def run_extraction(source_df, batch_size=4, debug=EXTRACT_DEBUG):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []
    stats = {
        "chunks_in": len(source_df),
        "batches_total": 0,
        "batches_ok": 0,
        "batches_failed": 0,
        "items_returned": 0,
        "relations_raw": 0,
        "relations_kept": 0,
        "skipped_chunk_id": 0,
        "skipped_empty_entity": 0,
        "skipped_node_type": 0,
        "skipped_relation": 0,
        "chunk_id_mismatches": [],
        "bad_schema_samples": [],
        "bad_relation_samples": [],
        "sample_response": None,
    }

    if debug:
        print(
            f"[NER+RE debug] provider={EXTRACT_PROVIDER!r} model={EXTRACT_MODEL!r} | "
            f"chunks={len(source_df)} | batch_size={batch_size}"
        )

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        stats["batches_total"] += 1
        try:
            obj, _ = extract_batch(batch)
            stats["batches_ok"] += 1
            if stats["sample_response"] is None and debug:
                stats["sample_response"] = obj
        except Exception as e:
            stats["batches_failed"] += 1
            errors.append({
                "start": start,
                "batch_size": len(batch),
                "chunk_ids": batch["chunk_id"].tolist()[:2],
                "error": str(e),
                "error_type": type(e).__name__,
            })
            continue

        stats["items_returned"] += len(obj.get("items", []))
        for item in obj.get("items", []):
            _parse_relations_from_item(item, meta, stats, triples)

        if EXTRACT_BATCH_SLEEP and start + batch_size < len(source_df):
            time.sleep(EXTRACT_BATCH_SLEEP)

    extraction_debug_stats = stats
    if debug:
        _print_extraction_debug(stats, errors)
    return pd.DataFrame(triples), pd.DataFrame(errors), extraction_debug_stats

def _print_extraction_debug(stats, errors):
    print("\n=== NER+RE DEBUG SUMMARY ===")
    print(
        f"batches: ok={stats['batches_ok']}/{stats['batches_total']} | "
        f"failed={stats['batches_failed']}"
    )
    print(
        f"items returned by LLM={stats['items_returned']} | "
        f"relations raw={stats['relations_raw']} | kept={stats['relations_kept']}"
    )
    print(
        "filtered out -> "
        f"chunk_id mismatch={stats['skipped_chunk_id']} | "
        f"empty entity={stats['skipped_empty_entity']} | "
        f"bad node type={stats['skipped_node_type']} | "
        f"bad relation={stats['skipped_relation']}"
    )

    if errors:
        print(f"\nAPI errors: {len(errors)} (showing first {EXTRACT_LOG_FIRST_N_ERRORS})")
        err_df = pd.DataFrame(errors).head(EXTRACT_LOG_FIRST_N_ERRORS)
        display(err_df)
        top = pd.Series([e["error"] for e in errors]).value_counts().head(3)
        print("Top error messages:")
        for msg, cnt in top.items():
            print(f"  [{cnt}x] {msg[:200]}")

    if stats["chunk_id_mismatches"]:
        print("\nchunk_id mismatch samples (LLM trả ID khác INPUT -> bị bỏ):")
        display(pd.DataFrame(stats["chunk_id_mismatches"]))
    if stats["bad_relation_samples"]:
        print("\nRelation không thuộc allowlist:")
        display(pd.DataFrame(stats["bad_relation_samples"]))
    if stats["bad_schema_samples"]:
        print("\nNode type không hợp lệ:")
        display(pd.DataFrame(stats["bad_schema_samples"]))

    if stats["relations_kept"] == 0:
        print("\n⚠️ 0 triples kept — gợi ý kiểm tra:")
        print(f"  1) EXTRACT_PROVIDER={EXTRACT_PROVIDER!r} EXTRACT_MODEL={EXTRACT_MODEL!r}")
        print("  2) extraction_errors_df có rate limit / auth error?")
        print("  3) chunk_id mismatch? (LLM rút gọn URL)")
        print("  4) Text quá ngắn / prompt quá conservative?")
        if stats["sample_response"] is not None:
            print("\nSample JSON từ batch đầu tiên thành công:")
            print(json.dumps(stats["sample_response"], ensure_ascii=False, indent=2)[:2000])

def debug_extraction_one_batch(source_df=None, n=4):
    """Chạy thử 1 batch nhỏ để debug nhanh (không loop 400 chunk)."""
    src = (source_df if source_df is not None else extraction_source).head(n).copy()
    print(f"Debug 1 batch | n={len(src)} | provider={EXTRACT_PROVIDER!r} model={EXTRACT_MODEL!r}")
    triples_df, errors_df, stats = run_extraction(src, batch_size=len(src), debug=True)
    return triples_df, errors_df, stats

raw_triples_df, extraction_errors_df, extraction_debug_stats = run_extraction(extraction_source)
display(raw_triples_df.head())
if len(extraction_errors_df):
    display(extraction_errors_df.head())

[NER+RE debug] provider='openai' model='gpt-4o-mini' | chunks=406 | batch_size=4


NER+RE: 100%|██████████| 102/102 [08:22<00:00,  4.93s/it]


=== NER+RE DEBUG SUMMARY ===
batches: ok=102/102 | failed=0
items returned by LLM=299 | relations raw=261 | kept=245
filtered out -> chunk_id mismatch=0 | empty entity=0 | bad node type=3 | bad relation=13

Relation không thuộc allowlist:


,chunk_id,relation
0,https://bestmediainfo.com/2023/03/ias-provides-verification-solution-to-amazon-publisher-services-connections-market...,PROVIDED
1,https://www.rochesterfirst.com/why-roc/why-roc-it-insights-expands-in-rochester-providing-outsourced-technology-serv...,EXPANDS
2,https://www.fool.com/investing/2023/09/24/3-things-about-symbiotic-that-smart-investors-know/::c0000,GENERATES_REVENUE_FROM
3,https://www.finextra.com/pressarticle/96025/syndio-joins-london-stock-exchange-marketplace::c0000,JOINED
4,https://www.zawya.com/en/business/energy/apicorp-exits-investment-in-oil-services-company-ashtead-technology-eoch4zq...,EXITS



Node type không hợp lệ:


,chunk_id,source_type,target_type,relation
0,https://www.bleepingcomputer.com/news/security/us-govt-offers-10-million-bounty-for-info-on-clop-ransomware/::c0000,Technology,Company|Person|Technology,DEVELOPED
1,https://www.seattlepi.com/business/article/integrated-electrical-services-fiscal-q3-18278847.php::c0000,Company,N/A,FOUNDED
2,https://www.reuters.com/technology/microsoft-says-early-june-service-outages-were-cyberattacks-2023-06-18/::c0000,Company,N/A,DEVELOPED


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,onsemi,Company,PARTNERED_WITH,Sineng Electric,Company,https://www.businesswire.com/news/home/20230515005855/en/onsemi-and-Sineng-Electric-Spearhead-the-Development-of-Sus...,2023-05-16,onsemi today announced that Sineng Electric will integrate onsemi EliteSiC silic,1.00
1,Ceinsys Tech Ltd,Company,FOUNDED,25 Years,Technology,https://www.devdiscourse.com/article/technology/2518545-ceinsys-tech-ltd-a-specialized-gis-mobility-engineering-serv...,2023-07-11,Ceinsys Tech Ltd celebrates 25 Years of Enhancing Possibilities,0.90
2,Walt Disney Co.,Company,LEADS,Bob Iger,Person,https://apnews.com/article/disney-disney-subscribers-loss-9b954f969e3c15fc18cd0054e5e7f6dd::c0000,2023-08-10,Walt Disney Co. CEO Bob Iger vowed to make Disney's streaming services profitable,0.95
3,Sojern,Company,ACQUIRED,VenueLytics,Company,https://skift.com/2023/07/11/sojern-expands-into-new-hotel-tech-via-acquisition-heres-the-thinking/::c0000,2023-07-11,Sojern has acquired VenueLytics a platform that provides guest management and communications software,0.90
4,dynaCERT,Company,DEVELOPED,HydraGEN™ Technology,Technology,https://financialpost.com/pmn/business-wire-news-releases-pmn/dynacerts-hydragen-technology-to-be-featured-by-the-ci...,2023-09-18,dynaCERT’s HydraGEN™ Carbon Emission Reduction Technology line of commercial products,1.00


In [10]:
print(raw_triples_df)

                    source_raw source_type        relation  \
0                       onsemi     Company  PARTNERED_WITH   
1             Ceinsys Tech Ltd     Company         FOUNDED   
2              Walt Disney Co.     Company           LEADS   
3                       Sojern     Company        ACQUIRED   
4                     dynaCERT     Company       DEVELOPED   
..                         ...         ...             ...   
240             UPS Healthcare     Company       DEVELOPED   
241               Aran Azarzar      Person           LEADS   
242  LinTech Pragmatics JV LLC     Company         FOUNDED   
243                        IBM     Company     INVESTED_IN   
244                      Intel     Company     INVESTED_IN   

                                    target_raw target_type  \
0                              Sineng Electric     Company   
1                                     25 Years  Technology   
2                                     Bob Iger      Person   
3      

## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [12]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1948.70it/s]


,type,left,right,similarity,decision
0,Company,Information Services Group Inc.,Information Services Group,0.917150,MERGE_VECTOR
1,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
2,Company,L&T Technology Services,L&T Technology Services Limited,0.925773,MERGE_VECTOR
3,Technology,X90A,X90,0.920579,MERGE_VECTOR


In [13]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [14]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 402, 'edges': 243, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,cc9c6ee3857729e221d3f6de,ServiceNow,Company,13
1,fb0f4df56fab164ec48722f0,Microsoft,Company,6
2,6a8e935df8fdc5e504778f02,workers,Person,4
3,f3f32693adef317dca914a90,Sensormatic Solutions,Company,4
4,773eeb9b7cc008bff365fcdd,OpenAI,Company,4
5,0e132222d1315530d6efca52,Google,Company,3
6,33e26345a3f61f0bc587821f,Hyperion Defense Solutions,Company,3
7,7f8f377f2d9ed5bac3b1579c,JFrog,Company,3
8,8f3d4914bf480bdc62e88d80,Ceinsys Tech Ltd,Company,3
9,b79293f3033084556da650c0,Information Services Group,Company,3


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [15]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches: 100%|██████████| 12/12 [00:24<00:00,  2.07s/it]

Flat vectors: 1503


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [24]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = llm_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""", provider=GENERATE_PROVIDER, model=GENERATE_MODEL)
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [17]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [25]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = llm_chat(
        [{"role": "system", "content": ANSWER_SYSTEM},
         {"role": "user", "content": prompt}],
        provider=GENERATE_PROVIDER,
        model=GENERATE_MODEL,
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [19]:
#@title 4.1 — 5 câu Golden starter
# LOCAL RUN: golden dataset lưu trong data/ để khớp cấu trúc bài nộp
os.makedirs("data", exist_ok=True)
GOLDEN_PATH = "data/graphrag_golden_50_first5000_detailed.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,difficulty,question,reference_answer,reference_evidence,evidence_row_ids_0based,evidence_urls_json,expected_hops,seed_entities,required_relations,adversarial_dimension,gold_reasoning,scoring_notes,source_scope
0,G5000-26,multi-hop,hard,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,row 2532 (2023-07-26 20:19:00): Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft...,"[2532, 2537]","[""https://www.reuters.com/technology/amazon-has-drawn-thousands-try-its-ai-service-competing-with-microsoft-google-2...",2,"[""Amazon"", ""Cohere""]","[""PROVIDES_ACCESS_TO"", ""DEVELOPED""]",Duplicate coverage + detail union,Merge the duplicate Amazon reports and retain non-conflicting details from each.,Need Cohere plus the customer-service-agent capability; clinical notes is an additional supported detail.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
1,G5000-27,cross-doc,hard,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,row 3357 (2023-06-01 13:16:00): 3 Best Cloud Stocks to Buy in June | row 2905 (2023-06-14 08:47:00): Exclusive: Amaz...,"[3357, 2905]","[""https://www.fool.com/investing/2023/06/01/3-best-cloud-stocks-to-buy-in-june/"", ""https://www.reuters.com/technolog...",2,"[""AMD"", ""AWS""]","[""POWERS"", ""CONSIDERING""]",General-to-specific relation trap,Distinguish a general market statement from a specific vendor adoption claim.,Critical: do not infer AWS adopted the new AMD AI chips.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
2,G5000-28,multi-hop,hard,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud,[3395],"[""https://www.tmcnet.com/usubmit/2023/08/29/9871484.htm""]",3,"[""Google Cloud"", ""Meta"", ""Technology Innovation Institute"", ""Anthropic""]","[""HOSTS_MODEL_FROM"", ""PREANNOUNCED""]",Multi-entity model/provider mapping,Build separate provider->model mappings rather than a flat list.,All three providers and their models are required.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
3,G5000-29,cross-doc,hard,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",row 3380 (2023-07-21 13:01:00): The White House and big tech companies release commitments on managing AI | row 3330...,"[3380, 3330]","[""https://www.wfae.org/united-states-world/united-states-world/2023-07-21/the-white-house-and-big-tech-companies-rel...",2,"[""White House"", ""Google"", ""Meta"", ""OpenAI"", ""IBM"", ""Adobe"", ""Salesforce""]","[""COMMITTED_TO""]",Temporal participant-set expansion,Compare the named participant sets and shared commitment concept over time.,Do not claim the September companies were part of the July seven unless explicitly named.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
4,G5000-30,multi-hop,hard,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...",row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud | row 3380 (2023-

In [26]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        return openai_json(system, user, model=JUDGE_MODEL)[0]

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [29]:
#@title 4.3 — Evaluation runner + checkpoint
# LOCAL RUN: checkpoint lưu trong outputs/ (thư mục deliverable) thay vì /content (Colab)
os.makedirs("outputs", exist_ok=True)
CHECKPOINT = "outputs/graphrag_eval_checkpoint.csv"
EVAL_SLEEP = 1.0  # giây nghỉ giữa các câu hỏi (tránh rate limit)

def run_evaluation(golden_df):
    print(
        f"[Eval] generate: {GENERATE_PROVIDER}/{GENERATE_MODEL} | "
        f"judge: {JUDGE_PROVIDER}/{JUDGE_MODEL}"
    )
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
        if EVAL_SLEEP:
            time.sleep(EVAL_SLEEP)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.
[Eval] generate: openai/gpt-4o-mini | judge: openai/gpt-4o-mini


Evaluation: 100%|██████████| 25/25 [04:54<00:00, 11.79s/it]


,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,The external technology provider named in Amazon's July AI-service expansion is Hugging Face. Alongside this partner...,The external technology provider named in Amazon's July AI-service expansion is Hugging Face. The other new AI capab...,1,1,1,1,1,1,3.313177,1.643764,842,1126,"The candidate incorrectly identifies Hugging Face as the external technology provider instead of Cohere, which is th...","The candidate incorrectly identifies Hugging Face as the external technology provider instead of Cohere, which is ex...",0
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,"The graph should illustrate that while AMD currently powers multiple cloud services, there is an ongoing evaluation ...","The graph should clarify that while AMD currently powers multiple cloud services, the later Reuters report about AWS...",3,3,4,4,3,3,4.754856,1.845785,904,646,The candidate provides a reasonable explanation of the relationship between AMD's current role in cloud services and...,The candidate provides a reasonable explanation of how AMD's existing role in cloud services and AWS's consideration...,0
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,The provided context does not specify any model providers connected to Google Cloud Next '23 or the models associate...,The model providers connected to Google Cloud Next '23 in the selected data are:\n\n1. **ServiceNow**\n - Models/T...,1,1,1,1,1,1,1.479030,1.816740,736,2321,The candidate fails to provide any information regarding the model providers connected to Google Cloud Next '23 or t...,The candidate answer fails to mention the correct model providers and their associated models as specified in the re...,0
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",Participation in White House AI commitments broadened from July to September 2023 as additional tech companies joine...,Participation in White House AI commitments broadened from July to September 2023 as more tech companies joined the ...,5,4,5,5,5,4,2.354653,2.063089,815,672,The candidate accurately summarizes the expansion of participation in White House AI commitments from July to Septem...,"The candidate provides a clear overview of the expansion in participation from July to September 2023, mentioning th...",0
4,G5000-30,multi-hop,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...",Meta appears in two different AI contexts in the provided data:\n\n1. **Commitments on Managing AI**: In this contex...,Meta appears in two different AI contexts in the selected data:\n\n1. **AI Management Commitments**: In this context...,2,2,3,2,2,2,3.719699,3.28

In [30]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
os.makedirs("outputs", exist_ok=True)
eval_results_df.to_csv("outputs/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("outputs/graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,2.273,2.364,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,2.636,3.091,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,2.182,2.273,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),2.307,1.825,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,793.091,723.727,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,2.500,2.500,Hai phương pháp gần nhau.
6,factoid,Faithfulness,3.000,3.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,2.500,2.500,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.260,1.381,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,762.500,714.500,GraphRAG không đắt hơn trong sample này.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [31]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': 'cc9c6ee3857729e221d3f6de', 'name': 'ServiceNow', 'degree': 13} fetched= 13


,type,left,right,similarity,decision
2,Company,L&T Technology Services,L&T Technology Services Limited,0.925773,MERGE_VECTOR
1,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
3,Technology,X90A,X90,0.920579,MERGE_VECTOR
0,Company,Information Services Group Inc.,Information Services Group,0.917150,MERGE_VECTOR


High-similarity rejected pairs:


,type,left,right,similarity,decision


In [32]:
entity_resolution_audit_df

,type,left,right,similarity,decision
0,Company,Information Services Group Inc.,Information Services Group,0.917150,MERGE_VECTOR
1,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
2,Company,L&T Technology Services,L&T Technology Services Limited,0.925773,MERGE_VECTOR
3,Technology,X90A,X90,0.920579,MERGE_VECTOR


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau